In [ ]:
import numpy as np
import pandas as pd
import torch
import random

from Conv1DAE import Conv1DAE, detect_anomalies_conv1dae, conv1dae_train
from optimize_models import latency_to_detection, get_basic_metrics, train_test_split_anomaly_sequence, optimize_conv1dae
from prepare_data import load_or_cache_twitter, load_or_cache_mit_bih, load_or_cache_bonn, \
    get_mit_bih_segments, get_twitter_segments, get_bonn_segments
from visualizations import heatmaps, segments_reconstruction

torch.manual_seed(0)
random.seed(0)

#### <center>Zbiór EEG Bonn</center>

Uznajemy, że zbiór E to outliery, a reszta:
- zdrowi oczy otwarte -> A,
- zdrowi oczy zamknięte -> B,
- pacjenci między napadami (zdrowa półkula) -> C,
- pacjenci między napadami (strefa padaczkowa) -> D,

są zdrowi.

In [ ]:
eeg_bonn_dataset = load_or_cache_bonn()

Przygotowany zbiór EEG-Bonn

Każda sekwencja ma przypisaną etykietę na podstawie przynależności do zbioru. Etykieta segmentu jest przypisana na podstawie stosunku liczby anomalii do normalnych próbek na poziomie sekwencji.

In [ ]:
bonn_overlaps = [0.25, 0.5, 0.75]
bonn_window_sizes = [2, 3, 5]

<center>Eksperymenty dla 1D Conv-AE</center>

In [ ]:
conv1dae_bonn_experiments, conv1dae_bonn_heatmap, conv1dae_bonn_training_params = [
    {
        latent: {
            s: {
                o: None for o in bonn_overlaps
            } for s in bonn_window_sizes
        } for latent in [8, 16]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_bonn_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            # Przygotowanie danych
            X_bonn, y_bonn, _ = get_bonn_segments(eeg_bonn_dataset, second, overlap)

            # Podział na train/test
            X_train_bonn, X_test_bonn, y_train_bonn, y_test_bonn = train_test_split_anomaly_sequence(X_bonn, y_bonn, random_state=42)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_bonn_study = optimize_conv1dae(
                X=X_train_bonn,
                y=y_train_bonn,
                latent=latent,
                dataset_name="EEG Bonn",
                n_trials=10
            )
            conv1dae_bonn_params = conv1dae_bonn_study.best_params
            conv1dae_bonn_train_params = {k: conv1dae_bonn_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_bonn_loss_params = {k: conv1dae_bonn_params[k] for k in ["alpha", "beta", "gamma"]}

            conv1dae_bonn_training_params[latent][second][overlap] = conv1dae_bonn_params

            # Konwersja z numpy do torch
            X_train_bonn, X_test_bonn = torch.from_numpy(X_train_bonn).to(dtype=torch.float32, device='cuda'), torch.from_numpy(X_test_bonn).to(dtype=torch.float32, device='cuda')

            # Autoenkoder
            conv1dae_bonn = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train_bonn = X_train_bonn[:, :, None, :]
            X_test_bonn = X_test_bonn[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae_bonn,
                data=X_train_bonn,
                **conv1dae_bonn_train_params
            )

            # Predykcje
            y_pred_bonn, y_scores_bonn, _ , reconstruction_bonn = detect_anomalies_conv1dae(
                model=conv1dae_bonn,
                data=X_test_bonn,
                threshold_percentile=80,
                **conv1dae_bonn_loss_params
            )

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_true=y_test_bonn.reshape(-1, 1).squeeze(),
                y_pred=y_pred_bonn.reshape(-1, 1).squeeze(),
                y_scores=y_scores_bonn.reshape(-1, 1).squeeze()
            )
            conv1dae_bonn_test_latencies = latency_to_detection(y_test_bonn, y_pred_bonn)
            conv1dae_bonn_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_bonn_test_latencies}
            conv1dae_bonn_heatmap[latent][second][overlap] = y_scores_bonn

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test_bonn.cpu().numpy().squeeze(2),
                X_pred=reconstruction_bonn,
                y_true=y_test_bonn,
                y_pred=y_pred_bonn.squeeze(),
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_bonn.png"
            )

In [ ]:
conv1dae_bonn_experiments

In [ ]:
conv1dae_bonn_training_params

In [ ]:
for latent in conv1dae_bonn_heatmap.keys():
    heatmaps(conv1dae_bonn_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze EEG Bonn", f"{latent}_conv1dae_heatmap_bonn")

#### <center>Zbiór MIT-BIH ECG</center>

Dzięki plikom atr mam dostęp, w którym momencie zostało zarejestrowane uderzenie serca. Dzięki temu mogę każdy z segmentów w sekwencjach oznaczać, ale może być wiele etykiet w segmencie. Aby przypisać czy jest outlierem zliczam wystąpienia "N" i pozostałych i porównuje, czego jest więcej.

In [ ]:
mit_bih = load_or_cache_mit_bih()

Przygotowany zbiór MIT BIH

Każdy segment posiada etykietę przypisaną na podstawie stosunku liczby uderzeń normalnych do arytmii.

In [ ]:
mit_bih_overlaps = [0.25, 0.5, 0.75]
mit_bih_window_sizes = [7, 10, 15]

<center>Eksperymenty dla 1D Conv-AE</center>

In [ ]:
conv1dae_mit_bih_experiments, conv1dae_mit_bih_heatmap, conv1dae_mit_bih_training_params = [
    {
        latent: {
            s: {
                o: None for o in mit_bih_overlaps
            } for s in mit_bih_window_sizes
        } for latent in [8, 16]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_mit_bih_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            torch.cuda.empty_cache()
            # Przygotowanie danych
            X_mit_bih, y_mit_bih, _ = get_mit_bih_segments(mit_bih, second, overlap)

            # Wybieramy te anomalie, które mają najmniej zanieczyszczonych segmentów
            anomaly_ratios = (y_mit_bih == -1).mean(axis=1)
            pseudo_y_mit_bih = np.where(anomaly_ratios < 0.2, 1, -1)

            # Podział na train/test
            X_train_mit_bih, X_test_mit_bih, y_train_mit_bih, y_test_mit_bih = train_test_split_anomaly_sequence(X_mit_bih, y_mit_bih, pseudo_label=pseudo_y_mit_bih)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_mit_bih_study = optimize_conv1dae(
                X=X_train_mit_bih,
                y=y_train_mit_bih,
                latent=latent,
                dataset_name="MIT BIH",
                n_trials=20,
                percentile=99
            )

            conv1dae_mit_bih_params = conv1dae_mit_bih_study.best_params
            conv1dae_mit_bih_train_params = {k: conv1dae_mit_bih_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_mit_bih_loss_params = {k: conv1dae_mit_bih_params[k] for k in ["alpha", "beta", "gamma"]}

            conv1dae_mit_bih_training_params[latent][second][overlap] = conv1dae_mit_bih_params

            # Konwersja z numpy do torch
            X_train_mit_bih, X_test_mit_bih = torch.from_numpy(X_train_mit_bih).to(dtype=torch.float32, device="cuda"), torch.from_numpy(X_test_mit_bih).to(dtype=torch.float32, device="cuda")

            # Autoenkoder
            conv1dae_mit_bih = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train_mit_bih = X_train_mit_bih[:, :, None, :]
            X_test_mit_bih = X_test_mit_bih[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae_mit_bih,
                data=X_train_mit_bih,
                **conv1dae_mit_bih_train_params
            )

            # Predykcje
            y_pred_mit_bih, y_scores_mit_bih, _, reconstruction_mit_bih = detect_anomalies_conv1dae(
                model=conv1dae_mit_bih,
                data=X_test_mit_bih,
                threshold_percentile=95,
                **conv1dae_mit_bih_loss_params
            )

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_test_mit_bih.reshape(-1, 1).squeeze(),
                y_pred_mit_bih.reshape(-1, 1).squeeze(),
                y_scores_mit_bih.reshape(-1, 1).squeeze()
            )
            conv1dae_mit_bih_test_latencies = latency_to_detection(y_test_mit_bih, y_pred_mit_bih)
            conv1dae_mit_bih_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_mit_bih_test_latencies}
            conv1dae_mit_bih_heatmap[latent][second][overlap] = y_scores_mit_bih

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test_mit_bih.cpu().numpy().squeeze(2),
                X_pred=reconstruction_mit_bih,
                y_true=y_test_mit_bih,
                y_pred=y_pred_mit_bih.squeeze(),
                n=2,
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_mit_bih.png"
            )

In [ ]:
conv1dae_mit_bih_experiments

In [ ]:
conv1dae_mit_bih_training_params

In [ ]:
for latent in conv1dae_mit_bih_heatmap.keys():
    heatmaps(conv1dae_mit_bih_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze MIT BIH", f"{latent}_conv1dae_heatmap_mit_bih")

#### <center>Zbiór Numenta Anomaly Benchmark (Twitter)</center>

In [ ]:
twitter = load_or_cache_twitter()

Przygotowany zbiór Twitter

Każdy segment posiada etykietę przypisaną na podstawie stosunku liczby anomalii do liczby prawidłowych próbek.

In [ ]:
twitter_window_sizes = [60 * 1, 60 * 2, 60 * 3]
twitter_overlaps = [0.25, 0.5, 0.75]

<center>Eksperymenty dla 1D Conv-AE</center>

In [ ]:
conv1dae_twitter_experiments, conv1dae_twitter_heatmap, conv1dae_twitter_training_params = [
    {
        latent: {
            s: {
                o: None for o in twitter_overlaps
            } for s in twitter_window_sizes
        } for latent in [8, 16]
    }  for _ in range(3)
]

for latent, seconds in conv1dae_twitter_experiments.items():
    for second, overlaps in seconds.items():
        for overlap in overlaps.keys():
            # Przygotowanie danych
            X_twitter, y_twitter, _ = get_twitter_segments(twitter, second, overlap)

            # Podział na train/test
            X_train_twitter, X_test_twitter, y_train_twitter, y_test_twitter = train_test_split_anomaly_sequence(X_twitter, y_twitter, random_state=42)

            # Optymalizacja parametrów Conv1DAE
            conv1dae_twitter_study = optimize_conv1dae(
                X=X_train_twitter,
                y=y_train_twitter,
                latent=latent,
                dataset_name="Twitter",
                n_trials=100,
                percentile=99
            )
            conv1dae_twitter_params = conv1dae_twitter_study.best_params
            conv1dae_twitter_train_params = {k: conv1dae_twitter_params[k] for k in ["lr", "epochs", "alpha", "beta", "gamma"]}
            conv1dae_twitter_loss_params = {k: conv1dae_twitter_params[k] for k in ["alpha", "beta", "gamma"]}
            conv1dae_twitter_training_params[latent][second][overlap] = conv1dae_twitter_params

            # Konwersja z numpy do torch
            X_train_twitter, X_test_twitter = torch.from_numpy(X_train_twitter).to(dtype=torch.float32, device="cuda"), torch.from_numpy(X_test_twitter).to(dtype=torch.float32, device="cuda")

            # Autoenkoder
            conv1dae_twitter = Conv1DAE(input_dim=1, latent_dim=latent)
            X_train_twitter = X_train_twitter[:, :, None, :]
            X_test_twitter = X_test_twitter[:, :, None, :]

            # Trening
            conv1dae_train(
                model=conv1dae_twitter,
                data=X_train_twitter,
                **conv1dae_twitter_train_params
            )

            # Predykcje
            y_pred_twitter, y_scores_twitter, _ ,reconstruction_twitter = detect_anomalies_conv1dae(
                model=conv1dae_twitter,
                data=X_test_twitter,
                **conv1dae_twitter_loss_params
            )

            # Zapisanie metryk
            experiment_basic_metrics = get_basic_metrics(
                y_test_twitter.reshape(-1, 1).squeeze(),
                y_pred_twitter.reshape(-1, 1).squeeze(),
                y_scores_twitter.reshape(-1, 1).squeeze()
            )
            conv1dae_twitter_test_latencies = latency_to_detection(y_test_twitter, y_pred_twitter)
            conv1dae_twitter_experiments[latent][second][overlap] = {**experiment_basic_metrics, **conv1dae_twitter_test_latencies}
            conv1dae_twitter_heatmap[latent][second][overlap] = y_scores_twitter

            # Rekonstrukcja sygnałów
            segments_reconstruction(
                X_test=X_test_twitter.cpu().numpy().squeeze(2),
                X_pred=reconstruction_twitter,
                y_true=y_test_twitter,
                y_pred=y_pred_twitter.squeeze(),
                n=2,
                title="",
                save_path=f"{latent}_{second}_{int(overlap * 100)}_reconstruction_analysis_twitter.png"
            )

In [ ]:
conv1dae_twitter_experiments

In [ ]:
conv1dae_twitter_training_params

Heatmapy

In [ ]:
for latent in conv1dae_twitter_heatmap.keys():
    heatmaps(conv1dae_twitter_heatmap.get(latent), f"Heatmapy anomalii dla Conv1D-AE z rozmiarem latent = {latent} na zbiorze Twitter", f"{latent}_conv1dae_heatmap_twitter")